# 🤖☕ RoboBarista – sichere Kännchenbewegung und echte Kamera-zu-Roboter-Rekonstruktion

Das ist ein **neues Experiment auf unserer Artly-inspirierten, sieben Stationen großen Kaffeeinsel**.

### Was passiert?

1. Ein vorab auf Gelenkgrenzen und simulierte Kollisionen geprüfter **zufälliger Testweg** wird mit festem Zufalls-Seed erzeugt. Er ist nicht von einem Menschen demonstriert.
2. Eine *separate virtuelle Milchkanne* macht diese Bewegung – nur ihre Bilder werden durch **drei echte MuJoCo-Kameras gerendert**.
3. Der **AprilTag-3-Detektor** sucht den Tag aus Pixeln. Unsere Mehrkamera-Optimierung schätzt daraus die 3D-Position und Rotation der Kanne. Fehlen zwei Ansichten, wird die Messung als **nicht gültig** markiert; keine Simulatorwahrheit zur Auffüllung.
4. Die gemessene Pose ist **alleiniger Sollwert für die FR5-IK**. Der FR5 bewegt mit seinen sechs Gelenken ein am Handgelenk befestigtes visuelles Kännchen.
5. **Alle Zwischenstellungen des Arms** werden vor dem Abspielen zusätzlich auf Roboter-Kontakt und mitgeführte Kannen-Hüllkörper geprüft. Bei Fehlern wird die Bewegung gesperrt.

### Was das noch nicht ist

**Keine echte menschliche Demonstration, keine dynamische Motor- oder Griffphysik, keine sichere physische Roboterfreigabe.** Die drei Kameras sind noch ideal kalibriert und haben synchronisierte virtuelle Bilder. Die Testbewegung ist bewusst klein und fern von heißen Geräten. Die separate Kanne in der Aufnahme wird zum Zwecke des Videos kinematisch bewegt.

**Film:** Die simulierte Demonstration links und die ausschließlich kameragespeiste FR5-Bewegung rechts werden nebeneinander dargestellt; darunter liegen die getrennten XYZ-Messkurven.

**Start:** Laufzeit → **Alle ausführen** (CPU, keine GPU nötig).


In [ ]:
# 1. Aktuelle Version aus unserem bestehenden GitHub holen
from pathlib import Path
import os,subprocess,sys
repo=Path('/content/robobarista-vision')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/Coffee-Tobi/robocasa.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
os.chdir(repo)
print('Projekt:',repo)


In [ ]:
# 2. Robotik- und Bildbibliotheken installieren
import subprocess,sys,os
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y',
               'libosmesa6-dev','libgl1','libglx-mesa0','libgl1-mesa-dri','ffmpeg'],check=True)
subprocess.run([sys.executable,'-m','pip','-q','install','-r','requirements.txt'],check=True)
print('Abhaengigkeiten installiert')


In [ ]:
# 3. Simulierten zufälligen Weg, Kamera-Bewegung und FR5-Replay ausführen
import subprocess,os,sys
environment=os.environ.copy()
environment['MUJOCO_GL']='egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
if environment['MUJOCO_GL']=='osmesa':
    environment['PYOPENGL_PLATFORM']='osmesa'
command=[sys.executable,'-u','-m','robobarista.camera_joint_replay',
         '--seed','14','--frames','13','--fps','5','--width','960','--height','720',
         '--output','runs/allowed_camera_replay']
print('Start:',environment['MUJOCO_GL'],flush=True)
process=subprocess.Popen(command,env=environment,cwd=str(repo),
             stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,
             errors='replace',bufsize=1)
for line in process.stdout:
    print(line,end='',flush=True)
code=process.wait()
print('Exit-Code:',code)
if code:
    print('Ein Sicherheits- oder Visionstest hat den Durchlauf unterbrochen.')
    print('Dann wurde KEINE ungueltige Bewegung als Erfolg ausgegeben.')


In [ ]:
# 4. Echtes RGB-D-Kameraexperiment UND danach FR5-Nachführung ansehen
from IPython.display import display,Video,Image
from pathlib import Path
import json,pandas as pd
out=Path(repo)/'runs/allowed_camera_replay'
film=out/'camera_vs_fr5_side_by_side.mp4'
fallback=out/'demo_and_replay.mp4'
gif=out/'demo_and_replay.gif'
if not film.is_file():film=fallback
print('Links: Kameras beobachten die virtuelle Demonstration')
print('Rechts: FR5 fuehrt die rekonstruierte Kamera-Bewegung aus')
if film.is_file():
    # IPython in Colab benötigt hier einen POSITIVEN Dateipfad.
    try:
        display(Video(str(film),embed=True,width=960))
    except (TypeError,ValueError,OSError):
        if gif.is_file():display(Image(filename=str(gif),format='gif'))
elif gif.is_file():
    display(Image(filename=str(gif),format='gif'))
else:
    print('Noch kein Film vorhanden – bitte die Fehlermeldung aus Schritt 3 lesen.')
chart=out/'camera_fr5_trajectory_comparison.png'
if chart.exists():
    display(Image(filename=str(chart)))
report=out/'summary.json'
if report.exists():
    s=json.loads(report.read_text())
    print('STATUS:',s['status'])
    print('Gueltige Kamera-Posen:',s['camera_valid_frames'],'/',s['frames'])
    print('Aus zwei oder mehr Kameraansichten:',s['camera_two_view_frames'],'/',s['frames'])
    print('Mit echten FR5-Gelenken nachgefahren:',s['replayed_frames'],'/',s['frames'])
    print('Max. kameragemessene vs. tatsächliche Bewegung (mm):',s['max_camera_vs_truth_mm'])
    print('Max. Ist-FR5 vs. Kameraziel (mm):',s['max_camera_to_executed_fr5_error_mm'])
    print('Arm-/Kannen-Sicherheitsprüfung vor jeder nachgefahrenen Stellung:',s['arm_link_and_payload_swept_preflight_on_every_generated_motion'])
else:
    print('Bericht fehlt. Bitte die Ausgabe aus Schritt 3 teilen.')


In [ ]:
# 5. Drei Trajektorien getrennt als Grafik – Quelle, Kamera, FR5
import json,numpy as np,matplotlib.pyplot as plt
truth_path=out/'ground_truth_evaluation_only.json'
cam_path=out/'camera_measurements.json'
robot_path=out/'robot_replay.json'
if all(x.exists() for x in (truth_path,cam_path,robot_path)):
    source=json.loads(truth_path.read_text())
    measured=json.loads(cam_path.read_text())
    executed=json.loads(robot_path.read_text())
    fig,axes=plt.subplots(3,1,figsize=(10,9),sharex=True)
    t=[f['time_s'] for f in source]
    for i,axis in enumerate('xyz'):
        axes[i].plot(t,[p['position_m'][i]*1000 for p in source],
                     label='Simulator-Wahrheit / nur Auswertung')
        axes[i].plot(t,[p['position_m'][i]*1000 if p['valid'] else float('nan')
                    for p in measured],':',label='aus Kamerabildern geschaetzt')
        axes[i].plot([p['time_s'] for p in executed],
                     [p['executed_position_m'][i]*1000 for p in executed],
                     '--',label='FR5 – vorwaertsgerichtete Kinematik')
        axes[i].set_ylabel(axis.upper()+' [mm]')
        axes[i].grid(alpha=.3)
    axes[0].legend()
    axes[-1].set_xlabel('Simulationszeit [s]')
    fig.suptitle('Der Soll-Ist-Kamera-Test – drei unterschiedliche Datenquellen')
    fig.tight_layout()
    plt.show()


In [ ]:
# 6. MP4, Kameradaten, Roboterwinkel und Messkurven als ZIP sichern
from IPython.display import FileLink,display
import zipfile
archive=out/'RoboBarista_Kamera_FR5_Test.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in out.glob('*'):
        if p.is_file() and p!=archive:
            z.write(p,p.name)
display(FileLink(str(archive)))


## Nächste Ausbaustufe

Aktuell ist der Zufallsweg schon **vorab in derselben Roboter-Kinematik erzeugt und als sicher gefiltert**. Dadurch ist er absichtlich klein und reproduzierbar. Der Roboter verwendet für seine Replay-IK trotzdem ausschließlich die **optisch rekonstruierten Werte**, nicht die Gelenkwinkel des ursprünglichen Weges.

Als nächstes müssen wir die größeren Zufahrten zu Mühle, Brühgruppe, Milchspender und Dampflanze prüfen. Dort sind derzeit mehrere kollidierende IK-Armlagen bekannt; ein einzelner erreichbarer Zielpunkt zählt **nicht** als kollisionsfreier Weg.

[Projektcode](https://github.com/Coffee-Tobi/robocasa) · [Sicherheits- und Geometrieplan](https://github.com/Coffee-Tobi/robocasa/blob/main/planning/WORKCELL_GEOMETRY_AND_SAFETY.md)
